# Agentic AI Systems — Worklet 13
# Research Grant Management System (apniLeap AURA)
## Lab 12 · Safety, Evaluation, and Governance — Approval, Evaluation, and Rollback

**Phase D · Governance & Capstone** &nbsp;|&nbsp; **Goal:** Add confidentiality, approval, evaluation, and rollback

---

**Description**

High-stakes institutional actions (submitting a grant proposal to a national agency,
allocating institute funds, signing binding legal terms) must never be executed autonomously
by an AI agent. This lab implements all six governance controls:
1. Least Privilege Permissions: The AI agent only holds draft-generation permissions;
   agency portal submission credentials are held exclusively by the Dean R&D.
2. Pre-Flight Diagnostic Evaluation Gate: The agent must pass a benchmark evaluation
   suite (accuracy >= 80%) before its proposals can be routed for review.
3. Named Human Approval Gate: Only users with the 'review:approve' permission
   (Dean R&D / Research Office) can authorize submission; PI self-approval is rejected.
4. Comprehensive Audit Trail: Every evaluation, rejection, approval, and edit is logged.
5. Immutable Releases & One-Click Rollback: Releases are versioned with SHA-256 checksums;
   any post-approval defect can be rolled back cleanly to a prior snapshot.
6. Privacy & Confidentiality: Rejection if raw patient/faculty private data is not scrubbed.

**Assignments included**

- Assignment 1: Cost-Category / Financial Clearance Gate (requiring finance approval for > 50L INR).
- Assignment 2: Mandatory Re-Evaluation Gate after every 5 proposals.

---

*Runs standalone on Python 3.10+ (Jupyter or Google Colab): no API key, no network, standard library only. Source script: [`scripts/lab12_safety_governance.py`](./scripts/lab12_safety_governance.py).*

### Cell 1 · Shared institutional seed data

`seed_data.py` is written to disk so the notebook runs standalone. It is the Python mirror of the course project's `backend/src/seed.js`: funding calls, literature corpus, policies, personnel rate card, users and the sanctioned-project portfolio. *(Collapse — shared by all labs.)*

In [1]:
%%writefile seed_data.py
"""
AURA Seed Data & Corpus -- Research Grant Management System
Corresponds to backend/src/seed.js, faculty.js, and projects.js
"""

INSTITUTION = {
    "id": "inst-mist",
    "name": "Meridian Institute of Science and Technology",
    "short": "MIST",
    "type": "university",
    "city": "Pune",
    "country": "IN",
    "researchOffice": "Research & Development Office",
    "signatory": "Dean, Research & Development",
}

USERS = [
    {
        "id": "usr-krishnan",
        "name": "Dr. Uma Mudengudi",
        "email": "uma.mudengudi@mist.edu.in",
        "role": "dean",
        "title": "Dean, Research & Development",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback", "metrics:read"],
    },
    {
        "id": "usr-suneeta-v-budihal",
        "name": "Dr. Suneeta V Budihal",
        "email": "suneeta.v.budihal@mist.edu.in",
        "role": "pi",
        "title": "Associate Professor",
        "department": "SoECE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-satyadhyan-chickerur",
        "name": "Dr. Satyadhyan Chickerur",
        "email": "satyadhyan.chickerur@mist.edu.in",
        "role": "pi",
        "title": "Professor & Head",
        "department": "CSE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-research-office",
        "name": "Research & Development Office",
        "email": "grants.cell@mist.edu.in",
        "role": "research_office",
        "title": "Grants Cell & Compliance",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback"],
    },
]

RATE_CARD = {
    "institutionId": "inst-mist",
    "overheadPct": 15,
    "contingencyPct": 5,
    "salaryEscalationPct": 5,
    "equipmentMaintenancePct": 4,
    "effectiveFrom": "2026-04-01",
    "note": "Institutional default rates. Agency-published rates for a call always take precedence.",
}

PERSONNEL = [
    {"id": "per-uma", "name": "Dr. Uma Mudengudi", "role": "PI", "title": "Dean (R&D) & Professor", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "professor"},
    {"id": "per-suneeta", "name": "Dr. Suneeta V Budihal", "role": "PI", "title": "Faculty Member", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-satyadhyan", "name": "Dr. Satyadhyan Chickerur", "role": "PI", "title": "Faculty Member", "department": "CSE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-ra", "name": "Research Associate (to be appointed)", "role": "RA", "title": "Research Associate", "department": "Research Office", "monthlyCost": 48000, "defaultEffortPct": 100, "careerStage": "n/a"},
    {"id": "per-phd", "name": "Doctoral Scholar (to be appointed)", "role": "PhD", "title": "Doctoral Scholar", "department": "Research Office", "monthlyCost": 36000, "defaultEffortPct": 100, "careerStage": "n/a"},
]

POLICIES = [
    {
        "id": "pol-overhead",
        "title": "Institutional overhead and costing policy",
        "tags": ["budget", "overhead"],
        "text": "Overheads are charged at the rate published by the funding agency for the call. Where the agency publishes no rate, the institutional default of 15% of total direct costs applies. Overhead may not fund permanent faculty salaries.",
    },
    {
        "id": "pol-ethics",
        "title": "Human subjects research ethics policy",
        "tags": ["compliance", "ethics", "health"],
        "text": "Any proposal involving human participants, identifiable health data or clinical validation requires Institutional Ethics Committee approval before the award is activated. A conditional certificate may be attached at submission and must be converted to full approval before funds are released.",
    },
    {
        "id": "pol-data",
        "title": "Research data protection policy (DPDP-aligned)",
        "tags": ["compliance", "data", "privacy"],
        "text": "Proposals collecting personal data must include a Data Management Plan covering lawful basis, de-identification, storage location, retention period and a named data steward. Cross-border transfer of health data requires a documented risk assessment.",
    },
    {
        "id": "pol-ai",
        "title": "Disclosure of AI-assisted writing",
        "tags": ["compliance", "ai", "authorship"],
        "text": "AI-assisted drafting must be disclosed in the proposal cover declaration and in the institutional submission record. AI-generated text is a draft until a named faculty member reviews and accepts it; AI tools may not be listed as authors or investigators.",
    },
    {
        "id": "pol-coi",
        "title": "Conflict of interest declaration",
        "tags": ["compliance", "coi"],
        "text": "Every investigator must declare financial interests, advisory roles and institutional relationships that could reasonably be perceived to influence the research. Declarations are collected by the Research Office and retained for the audit period.",
    },
    {
        "id": "pol-signoff",
        "title": "Authorised signatory policy",
        "tags": ["compliance", "approval"],
        "text": "Only the Dean, Research & Development, or a delegated Research Office signatory, may authorise submission to a funding agency. The authorising officer must record the decision, the version reviewed, and any conditions attached to approval.",
    },
]

COMPLIANCE_RULES = [
    {"id": "rule-sections", "kind": "sections", "appliesTo": "all", "severity": "blocking", "requirement": "Every section required by the call is present and non-empty.", "evidence": "Call required-section list vs draft section index."},
    {"id": "rule-pages", "kind": "pageLimit", "appliesTo": "all", "severity": "blocking", "requirement": "The narrative stays within the page limit published for the call.", "evidence": "Word count converted at 500 words per page."},
    {"id": "rule-budget-ceiling", "kind": "budgetCeiling", "appliesTo": "all", "severity": "blocking", "requirement": "The requested total does not exceed the agency ceiling for this call.", "evidence": "Budget total vs call award maximum."},
    {"id": "rule-equipment-cap", "kind": "equipmentCap", "appliesTo": "all", "severity": "blocking", "requirement": "Equipment does not exceed the percentage ceiling in the call guidelines.", "evidence": "Equipment subtotal / total requested."},
    {"id": "rule-salary-cap", "kind": "salaryCap", "appliesTo": "all", "severity": "blocking", "requirement": "Manpower costs stay within the call salary ceiling.", "evidence": "Personnel subtotal / total requested."},
    {"id": "rule-overhead", "kind": "overheadRate", "appliesTo": "all", "severity": "blocking", "requirement": "Overhead is charged at exactly the rate published for the call.", "evidence": "Overhead amount vs rate x direct costs."},
    {"id": "rule-ethics", "kind": "document", "docKey": "ethics", "appliesTo": "all", "severity": "blocking", "requirement": "Ethics clearance is attached where the call or the research involves human participants.", "evidence": "Document register vs call requirement."},
    {"id": "rule-coi", "kind": "document", "docKey": "coi", "appliesTo": "all", "severity": "blocking", "requirement": "Conflict-of-interest declarations are attached for all investigators.", "evidence": "Document register vs call requirement."},
    {"id": "rule-dmp", "kind": "document", "docKey": "dmp", "appliesTo": "all", "severity": "blocking", "requirement": "A Data Management Plan is attached where required.", "evidence": "Document register vs call requirement."},
    {"id": "rule-disclosure", "kind": "aiDisclosure", "appliesTo": "all", "severity": "blocking", "requirement": "AI-assisted drafting is disclosed in the submission declaration.", "evidence": "Institutional AI disclosure policy vs package declaration."},
]

DOCUMENT_REGISTER = [
    {"key": "ethics", "label": "Institutional Ethics Committee clearance", "status": "conditional", "note": "Conditional certificate available; full approval pending protocol review."},
    {"key": "coi", "label": "Conflict-of-interest declarations (all investigators)", "status": "attached", "note": "Signed by all investigators."},
    {"key": "dmp", "label": "Data Management Plan", "status": "attached", "note": "Deposited in the institutional DMP registry."},
    {"key": "budget-justification", "label": "Budget justification", "status": "attached", "note": "Generated by the Budget Planning Agent."},
    {"key": "csr-1", "label": "CSR-1 registration and company CSR policy", "status": "pending", "note": "Company secretarial certificate requested."},
    {"key": "consortium-agreement", "label": "Consortium agreement and partner letters", "status": "pending", "note": "Letters of intent collected; agreement under legal review."},
]

CALLS = [
    {
        "id": "call-serb-crg",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "Core Research Grant (CRG)",
        "title": "SERB Core Research Grant 2026 -- Individual Investigator",
        "fundingType": "government",
        "domains": ["ai-ml", "energy", "materials", "health", "climate"],
        "tags": ["artificial intelligence", "machine learning", "federated learning", "sustainable energy", "materials science", "medical imaging"],
        "deadline": "2026-12-15",
        "budgetMin": 3000000,
        "budgetMax": 7500000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-serb-crg#g1", "section": "submission", "text": "Proposals must be routed through the institutional research office; individual submissions are not accepted."},
            {"id": "call-serb-crg#g2", "section": "budget", "text": "The total request may not exceed INR 75,00,000 for a duration of up to 36 months."},
            {"id": "call-serb-crg#g3", "section": "budget", "text": "Equipment may not exceed 40% of the total requested budget; requests above this ceiling require a written justification from the Head of Institution."},
            {"id": "call-serb-crg#g4", "section": "budget", "text": "Overheads are payable at 15% of total direct costs."},
            {"id": "call-serb-crg#g5", "section": "documents", "text": "A Data Management Plan and a conflict-of-interest declaration are mandatory at submission."},
            {"id": "call-serb-crg#g6", "section": "format", "text": "The narrative may not exceed 25 pages excluding references and the budget annexure."},
        ],
    },
    {
        "id": "call-icmr-adhoc",
        "agency": "ICMR",
        "agencyFull": "Indian Council of Medical Research",
        "program": "Ad-hoc Research Project",
        "title": "ICMR Ad-hoc Research Project 2026 -- Diagnostics & Public Health",
        "fundingType": "government",
        "domains": ["health", "ai-ml", "public-health"],
        "tags": ["diabetic retinopathy", "medical imaging", "rural health", "screening", "federated learning", "digital health", "primary care"],
        "deadline": "2026-12-05",
        "budgetMin": 2000000,
        "budgetMax": 6000000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 50, "equipmentCapPct": 30, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-icmr-adhoc#g1", "section": "eligibility", "text": "Applicants must hold a regular position in a recognised institution or hospital and demonstrate access to the study population."},
            {"id": "call-icmr-adhoc#g2", "section": "budget", "text": "The total request must fall between INR 20,00,000 and INR 60,00,000; equipment is capped at 30%."},
            {"id": "call-icmr-adhoc#g3", "section": "ethics", "text": "Institutional Ethics Committee approval is mandatory for any study involving human participants or their data."},
            {"id": "call-icmr-adhoc#g4", "section": "data", "text": "A Data Management Plan describing de-identification and retention is required; identifiable health data may not leave the institution without a documented risk assessment."},
            {"id": "call-icmr-adhoc#g5", "section": "deliverables", "text": "The proposal must state measurable clinical or public-health outcomes for the funding period."},
        ],
    },
    {
        "id": "call-tata-csr",
        "agency": "Tata Trusts",
        "agencyFull": "Tata Trusts -- Corporate Social Responsibility",
        "program": "CSR Research Partnership",
        "title": "Tata Trusts CSR Research Partnership 2026 -- Health & Education",
        "fundingType": "industry",
        "domains": ["health", "education", "public-health"],
        "tags": ["community health", "digital health", "education technology", "rural outreach", "screening"],
        "deadline": "2026-11-15",
        "budgetMin": 2500000,
        "budgetMax": 10000000,
        "durationMonths": 24,
        "pageLimit": 20,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital", "ngo"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 8, "salaryCapPct": 55, "equipmentCapPct": 25, "travelCapPct": 10, "contingencyPct": 3},
        "guidelines": [
            {"id": "call-tata-csr#g1", "section": "compliance", "text": "The activity must fall within Schedule VII of the Companies Act 2013 and the implementing partner must hold a valid CSR-1 registration."},
            {"id": "call-tata-csr#g2", "section": "budget", "text": "Grants range from INR 25,00,00,000 to INR 1,00,00,000 over 24 months; overheads are capped at 8%."},
            {"id": "call-tata-csr#g3", "section": "reporting", "text": "Quarterly outcome reporting against agreed indicators is a condition of release of the second tranche."},
        ],
    },
    {
        "id": "call-eu-horizon",
        "agency": "EU Horizon Europe",
        "agencyFull": "European Commission -- Horizon Europe",
        "program": "Cluster 4: Digital, Industry and Space",
        "title": "Horizon Europe 2027 -- Trustworthy AI for Health Systems (RIA)",
        "fundingType": "international",
        "domains": ["ai-ml", "health", "digital"],
        "tags": ["trustworthy ai", "federated learning", "health systems", "digital health", "data governance", "open science"],
        "deadline": "2027-01-20",
        "budgetMin": 18000000,
        "budgetMax": 45000000,
        "durationMonths": 36,
        "pageLimit": 45,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": True, "minPartners": 3, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 25, "salaryCapPct": 40, "equipmentCapPct": 30, "travelCapPct": 15, "contingencyPct": 0},
        "guidelines": [
            {"id": "call-eu-horizon#g1", "section": "eligibility", "text": "Proposals require a consortium of at least three independent legal entities from three different countries."},
            {"id": "call-eu-horizon#g2", "section": "budget", "text": "Personnel costs may not exceed 40% and equipment 30% of the total eligible cost; a 25% flat rate covers indirect costs."},
            {"id": "call-eu-horizon#g3", "section": "open-science", "text": "Open science practices, FAIR data management and a gender and inclusion plan are evaluated under the impact criterion."},
        ],
    },
    {
        "id": "call-serb-supra-expired",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "SUPRA",
        "title": "SERB SUPRA 2026 -- Closed",
        "fundingType": "government",
        "domains": ["ai-ml", "health"],
        "tags": ["artificial intelligence", "medical imaging"],
        "deadline": "2026-06-30",
        "budgetMin": 1000000,
        "budgetMax": 3000000,
        "durationMonths": 24,
        "status": "closed",
        "verified": True,
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [{"id": "call-serb-supra-expired#g1", "section": "status", "text": "This call closed on 30 June 2026 and is retained for reference only."}],
    },
    {
        "id": "call-unverified-sample",
        "agency": "Unknown Aggregator",
        "agencyFull": "Third-party opportunity aggregator",
        "program": "Sample Aggregator Call",
        "title": "Unverified Grant Opportunity 2027",
        "fundingType": "government",
        "domains": ["engineering"],
        "tags": ["engineering"],
        "deadline": "2027-02-01",
        "budgetMin": 800000,
        "budgetMax": 2500000,
        "durationMonths": 24,
        "verified": False,
        "status": "unverified",
        "eligibility": {"careerStages": ["assistant"], "institutionTypes": ["college"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 10, "salaryCapPct": 40, "equipmentCapPct": 35, "travelCapPct": 12, "contingencyPct": 5},
        "guidelines": [{"id": "call-unverified#g1", "section": "status", "text": "Listing could not be confirmed against an official agency notification."}],
    },
]

LITERATURE = [
    {
        "id": "lit-001",
        "title": "Federated learning for diabetic retinopathy screening across rural clinics",
        "authors": ["S. Banerjee", "R. Iyer", "M. Kulkarni"],
        "year": 2024,
        "venue": "Journal of Medical AI Research",
        "tags": ["federated learning", "diabetic retinopathy", "rural health", "medical imaging", "screening"],
        "abstract": "A four-site federated study reports comparable sensitivity to centralised training while keeping 1.2 million fundus images inside participating clinics.",
        "citations": 61,
        "doi": "10.5555/aura.demo.001",
    },
    {
        "id": "lit-002",
        "title": "Deep learning detection of referable diabetic retinopathy in primary care",
        "authors": ["L. Fernandes", "A. Gupta"],
        "year": 2023,
        "venue": "Ophthalmology Informatics",
        "tags": ["diabetic retinopathy", "medical imaging", "primary care", "screening", "deep learning"],
        "abstract": "A prospective primary-care evaluation of 8,400 patients reports 92% sensitivity and 88% specificity for referable retinopathy.",
        "citations": 148,
        "doi": "10.5555/aura.demo.002",
    },
    {
        "id": "lit-003",
        "title": "Privacy-preserving federated learning in healthcare: a systematic review",
        "authors": ["H. Park", "J. Okafor", "N. Singh"],
        "year": 2024,
        "venue": "Health Data Science Review",
        "tags": ["federated learning", "privacy", "health systems", "data governance"],
        "abstract": "Reviews 74 studies and finds that differential privacy and secure aggregation are rarely combined in clinical deployments.",
        "citations": 96,
        "doi": "10.5555/aura.demo.003",
    },
    {
        "id": "lit-004",
        "title": "Non-mydriatic fundus photography in resource-limited settings",
        "authors": ["P. Adjei", "T. Nguyen"],
        "year": 2022,
        "venue": "Global Eye Health",
        "tags": ["rural health", "screening", "medical imaging", "primary care"],
        "abstract": "Image quality is the dominant failure mode for non-mydriatic cameras operated by trained community health workers.",
        "citations": 57,
        "doi": "10.5555/aura.demo.004",
    },
    {
        "id": "lit-005",
        "title": "Teleophthalmology screening pathways in low-resource districts",
        "authors": ["K. Rao", "D. Bhattacharya"],
        "year": 2023,
        "venue": "Indian Journal of Public Health",
        "tags": ["screening", "rural health", "health systems", "telemedicine"],
        "abstract": "Hub-and-spoke teleophthalmology reduced time-to-diagnosis from 26 to 9 days across six districts.",
        "citations": 73,
        "doi": "10.5555/aura.demo.005",
    },
]

HISTORY = [
    {
        "id": "hist-retina-2024",
        "title": "AI-assisted retinal screening for diabetic retinopathy in Pune district",
        "agency": "ICMR",
        "year": 2024,
        "outcome": "not-funded",
        "amount": 5600000,
        "tags": ["diabetic retinopathy", "screening", "rural health", "medical imaging"],
        "abstract": "Proposed a three-site pilot of automated retinal grading with referral support for 12,000 patients.",
        "lessons": "Reviewers questioned the missing ethics clearance and the absence of a site-level data-governance plan. Both are now prerequisites.",
    },
    {
        "id": "hist-fl-testbed-2023",
        "title": "Federated learning testbed for multi-hospital medical imaging",
        "agency": "SERB",
        "year": 2023,
        "outcome": "funded",
        "amount": 6200000,
        "tags": ["federated learning", "medical imaging", "privacy", "testbed"],
        "abstract": "Built a three-hospital federated learning testbed with secure aggregation and site-level audit logging.",
        "lessons": "Delivered the testbed; clinical validation was out of scope and remains the open gap.",
    },
]


Writing seed_data.py


### Cell 2 · Imports and fixtures

In [2]:
import hashlib
import json
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, Any, List, Optional
from seed_data import USERS, INSTITUTION


# Benchmark evaluation accuracy for the diagnostic drafting agent
REGRESSION_SUITE_RESULT = {"passed": 9, "total": 10}  # 90% accuracy

### Cell 3 · `ProposalRelease` (class)

In [3]:
@dataclass
class ProposalRelease:
    version: int
    run_id: str
    checksum: str
    approved_by: str
    snapshot: Dict[str, Any]
    timestamp: str

### Cell 4 · `GovernedGrantAgent` (class)

In [4]:
class GovernedGrantAgent:
    """
    Enforces the six governance and safety controls.
    """
    permissions = {"can_write": "draft_only"}  # Control 1: Least Privilege

    def __init__(self):
        self.audit_log: List[Dict[str, Any]] = []
        self.evaluated: bool = False
        self.proposals_processed_since_eval: int = 0
        self.version_history: List[ProposalRelease] = []

    def evaluate_readiness(self) -> bool:
        """Control 2: Diagnostic Evaluation Gate (must pass >= 80% threshold)."""
        pass_rate = REGRESSION_SUITE_RESULT["passed"] / REGRESSION_SUITE_RESULT["total"]
        self.evaluated = pass_rate >= 0.8
        self._log("evaluation_gate", {"pass_rate": pass_rate, "gate_passed": self.evaluated})
        self.proposals_processed_since_eval = 0
        return self.evaluated

    def _log(self, step: str, data: Dict[str, Any]):
        """Control 4: Audit Logging for all decision points."""
        self.audit_log.append({
            "step": step,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "data": data,
        })

    def propose_submission_package(self, run_id: str, title: str, budget_total: float,
                                   rollback_plan: Optional[str], privacy_scrubbed: bool,
                                   finance_approved: bool = True) -> Dict[str, Any]:
        """Validates safety evidence before drafting an approval request."""
        # Checkpoint: Evaluation Gate
        if not self.evaluated:
            self._log("propose_rejected", {"reason": "agent_not_evaluated"})
            return {"rejected": True, "reason": "Evaluation gate not passed; agent cannot propose packages."}

        # Checkpoint: Periodic Re-evaluation (Assignment 2)
        if self.proposals_processed_since_eval >= 5:
            self._log("propose_rejected", {"reason": "mandatory_reevaluation_required"})
            return {"rejected": True, "reason": "5 proposals processed since last evaluation; re-evaluation required."}

        # Checkpoint: Least Privilege assertion
        if self.permissions["can_write"] != "draft_only":
            raise PermissionError("Security Violation: Agent must never hold production portal submission rights.")

        # Checkpoint 6: Privacy Scrubbing
        if not privacy_scrubbed:
            self._log("propose_rejected", {"reason": "privacy_unscrubbed_patient_data"})
            return {"rejected": True, "reason": "Diagnostic input contains unscrubbed patient data; rejecting."}

        # Checkpoint 5: Mandatory Rollback Plan
        if not rollback_plan:
            self._log("propose_rejected", {"reason": "missing_rollback_plan"})
            return {"rejected": True, "reason": "No rollback contingency plan defined; rejecting."}

        # Assignment 1: Financial threshold clearance
        if budget_total > 5000000 and not finance_approved:
            self._log("propose_rejected", {"reason": "finance_clearance_missing"})
            return {"rejected": True, "reason": "Grants exceeding 50 Lakh INR require explicit Finance Committee clearance."}

        # Control 3: Human Approval Request creation
        self.proposals_processed_since_eval += 1
        approval_request = {
            "run_id": run_id,
            "title": title,
            "budget_total": budget_total,
            "rollback_plan": rollback_plan,
            "privacy_scrubbed": privacy_scrubbed,
            "category": "agency_grant_submission",
            "requires_human_approval": True,
            "status": "AWAITING_DEAN_APPROVAL",
        }
        self._log("approval_requested", approval_request)
        return approval_request

    def authorize_and_publish(self, approval_request: Dict[str, Any], actor: Dict[str, Any],
                              decision: str, notes: str = "") -> Dict[str, Any]:
        """
        Human gate: Evaluates actor permissions.
        Only users with 'review:approve' can authorize publication.
        """
        self._log("human_decision_received", {"actor": actor["name"], "decision": decision})

        # Role-based Access Control Check
        if "review:approve" not in actor.get("permissions", []):
            self._log("authorization_blocked", {"actor": actor["name"], "reason": "insufficient_permissions"})
            raise PermissionError(f"Access Denied: Actor '{actor['name']}' ({actor['role']}) does not hold 'review:approve' permission.")

        if decision != "approved":
            return {"published": False, "status": "changes_requested", "notes": notes}

        # Proceed with Release & Cryptographic Versioning
        new_version = len(self.version_history) + 1
        serialized = json.dumps(approval_request, sort_keys=True)
        checksum = hashlib.sha256(serialized.encode()).hexdigest()

        release = ProposalRelease(
            version=new_version,
            run_id=approval_request["run_id"],
            checksum=checksum,
            approved_by=actor["name"],
            snapshot=approval_request,
            timestamp=datetime.now().isoformat(),
        )
        self.version_history.append(release)

        self._log("published_version", {"version": new_version, "checksum": checksum, "approver": actor["name"]})

        return {
            "published": True,
            "version": new_version,
            "checksum": checksum,
            "approver": actor["name"],
            "status": "PUBLISHED_TO_PORTAL_BY_RESEARCH_OFFICE",
            "auto_submitted_by_agent": False,  # Agent never submits directly!
        }

    def rollback(self, target_version: int, actor: Dict[str, Any]) -> Dict[str, Any]:
        """
        One-click rollback: Restores a previous version snapshot and records a new version entry.
        """
        if "version:rollback" not in actor.get("permissions", []):
            raise PermissionError(f"Access Denied: Actor '{actor['name']}' lacks 'version:rollback' permission.")

        target = next((v for v in self.version_history if v.version == target_version), None)
        if not target:
            raise ValueError(f"Version {target_version} not found in immutable history.")

        # Publishing a rollback creates a new version with the restored snapshot
        restored_version = len(self.version_history) + 1
        new_release = ProposalRelease(
            version=restored_version,
            run_id=target.run_id,
            checksum=target.checksum,
            approved_by=actor["name"],
            snapshot=target.snapshot,
            timestamp=datetime.now().isoformat(),
        )
        self.version_history.append(new_release)
        self._log("rollback_executed", {"from_version": target_version, "new_version": restored_version, "actor": actor["name"]})

        return {
            "status": "rolled_back",
            "restored_from_version": target_version,
            "new_published_version": restored_version,
            "checksum": target.checksum,
        }

### Cell 5 · `run_assignment_1_cost_category_gate` (function)

In [5]:
# ==============================================================================
# Graded Assignments
# ==============================================================================

def run_assignment_1_cost_category_gate(agent: GovernedGrantAgent):
    """
    Assignment 1 -- Cost-Category Control:
    A grant exceeding 50 Lakh INR must have finance_approved=True.
    Adversarial test with missing flag must be rejected.
    """
    print("\n--- [Assignment 1] Cost-Category Financial Clearance Gate ---")
    res = agent.propose_submission_package(
        run_id="RUN-2026-0005",
        title="High-End AI Supercomputer Grant",
        budget_total=6500000,  # 65L > 50L threshold
        rollback_plan="Revert to prior draft v1",
        privacy_scrubbed=True,
        finance_approved=False,  # Missing clearance!
    )
    print(f"Adversarial Test (Missing Finance Clearance): {res}")
    assert res.get("rejected") is True, "Proposal should have been rejected for missing finance clearance!"
    print("Verification passed: Financial threshold control strictly enforced.")

### Cell 6 · `run_assignment_2_periodic_reevaluation` (function)

In [6]:
def run_assignment_2_periodic_reevaluation(agent: GovernedGrantAgent):
    """
    Assignment 2 -- Mandatory Re-Evaluation Gate After N Proposals:
    After processing 5 proposals, confirm the 6th is blocked until
    evaluate_readiness() is called again.
    """
    print("\n--- [Assignment 2] Mandatory Periodic Re-Evaluation Gate ---")
    # Simulate processing 5 proposals
    agent.proposals_processed_since_eval = 5

    res = agent.propose_submission_package(
        run_id="RUN-2026-0006",
        title="6th Consecutive Proposal",
        budget_total=3000000,
        rollback_plan="Revert to v1",
        privacy_scrubbed=True,
    )
    print(f"6th Proposal Attempt without re-evaluation: {res}")
    assert res.get("rejected") is True, "6th proposal should be blocked without re-evaluation!"

    # Re-evaluate
    agent.evaluate_readiness()
    print("Re-evaluated readiness benchmark: gate passed.")

    res_retry = agent.propose_submission_package(
        run_id="RUN-2026-0006",
        title="6th Consecutive Proposal",
        budget_total=3000000,
        rollback_plan="Revert to v1",
        privacy_scrubbed=True,
    )
    print(f"6th Proposal Attempt after re-evaluation: Status = {res_retry.get('status')}")
    assert res_retry.get("requires_human_approval") is True, "Should succeed after re-evaluation!"
    print("Verification passed: Periodic re-evaluation successfully enforced.")

### Cell 7 · Run the lab

The lab demo and both graded assignments, with the console captured for the checks below.

In [7]:
# Console capture: everything the lab prints is shown below AND kept in memory,
# so the acceptance checks in the last cell can assert on the real run.
import io
import sys


class ConsoleTee(io.StringIO):
    """Mirror stdout into a buffer while the lab runs."""

    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, *exc):
        sys.stdout = self._stdout
        return False

    def write(self, text):
        self._stdout.write(text)
        return super().write(text)


console = ConsoleTee()

with console:
    print("======================================================================")
    print("Lab 12: Safety, Evaluation, and Governance (Approval & Rollback)")
    print("======================================================================")

    agent = GovernedGrantAgent()

    # 1. Control 2: Pre-flight Evaluation Gate
    print("=== Control 2: Pre-Flight Evaluation Gate ===")
    eval_passed = agent.evaluate_readiness()
    print(f"Agent evaluated against benchmark test suite: {eval_passed}")

    # 2. Rejection Test 1: Missing Rollback Plan
    print("\n=== Attempt 1: Missing Rollback Plan (Must Reject) ===")
    rej1 = agent.propose_submission_package(
        run_id="RUN-2026-0001",
        title="Diabetic Retinopathy Grant",
        budget_total=4500000,
        rollback_plan=None,  # Missing!
        privacy_scrubbed=True,
    )
    print(f"Result: {rej1}")

    # 3. Rejection Test 2: Unscrubbed Patient Data
    print("\n=== Attempt 2: Unscrubbed Patient Data (Must Reject) ===")
    rej2 = agent.propose_submission_package(
        run_id="RUN-2026-0001",
        title="Diabetic Retinopathy Grant",
        budget_total=4500000,
        rollback_plan="Revert to v1",
        privacy_scrubbed=False,  # Raw clinical identifiers present!
    )
    print(f"Result: {rej2}")

    # 4. Valid Proposal drafted
    print("\n=== Attempt 3: Valid Proposal Drafted ===")
    valid_request = agent.propose_submission_package(
        run_id="RUN-2026-0001",
        title="Diabetic Retinopathy Grant",
        budget_total=4500000,
        rollback_plan="Revert to initial seed v1 on agency rejection",
        privacy_scrubbed=True,
    )
    print(f"Status: {valid_request['status']} (Run: {valid_request['run_id']})")

    # 5. Authorization Attempt 1: PI Attempting to Approve (Must Fail)
    print("\n=== Human Gate: PI Attempts Self-Approval (Must Reject) ===")
    pi_user = next(u for u in USERS if u["role"] == "pi")
    try:
        agent.authorize_and_publish(valid_request, actor=pi_user, decision="approved")
    except PermissionError as e:
        print(f"Blocked as expected: {e}")

    # 6. Authorization Attempt 2: Dean R&D Approves (Must Pass)
    print("\n=== Human Gate: Dean R&D Authorizes Publication ===")
    dean_user = next(u for u in USERS if u["role"] == "dean")
    release_info = agent.authorize_and_publish(valid_request, actor=dean_user, decision="approved")
    print(f"Published Version: v{release_info['version']} | Checksum: {release_info['checksum'][:16]}...")

    # 7. Rollback Demonstration
    print("\n=== One-Click Rollback Demonstration ===")
    rollback_info = agent.rollback(target_version=1, actor=dean_user)
    print(f"Rollback complete: {rollback_info}")

    # 8. Run Assignments
    run_assignment_1_cost_category_gate(agent)
    run_assignment_2_periodic_reevaluation(agent)

    # 9. Audit Log Inspection
    print("\n=== Full Governance Audit Trail (Sample) ===")
    for entry in agent.audit_log[:5]:
        print(f"  {entry['timestamp']} | Step: {entry['step']} | Data: {entry['data']}")

Lab 12: Safety, Evaluation, and Governance (Approval & Rollback)
=== Control 2: Pre-Flight Evaluation Gate ===
Agent evaluated against benchmark test suite: True

=== Attempt 1: Missing Rollback Plan (Must Reject) ===
Result: {'rejected': True, 'reason': 'No rollback contingency plan defined; rejecting.'}

=== Attempt 2: Unscrubbed Patient Data (Must Reject) ===
Result: {'rejected': True, 'reason': 'Diagnostic input contains unscrubbed patient data; rejecting.'}

=== Attempt 3: Valid Proposal Drafted ===
Status: AWAITING_DEAN_APPROVAL (Run: RUN-2026-0001)

=== Human Gate: PI Attempts Self-Approval (Must Reject) ===
Blocked as expected: Access Denied: Actor 'Dr. Suneeta V Budihal' (pi) does not hold 'review:approve' permission.

=== Human Gate: Dean R&D Authorizes Publication ===
Published Version: v1 | Checksum: 052bfac47104fa94...

=== One-Click Rollback Demonstration ===
Rollback complete: {'status': 'rolled_back', 'restored_from_version': 1, 'new_published_version': 2, 'checksum': '

### Cell 8 · Acceptance checks and deliverables

Each check asserts on the trace of the run above. The trace and the check report are exported to `lab12_artifacts/` (committed under `Course-Project/evidence/`).

In [8]:
import json
import re
from pathlib import Path

LAB = 12
CHECKS = [('Proposal without a rollback plan is rejected', 'No rollback contingency plan defined', 1),
    ('Unscrubbed patient data is rejected', 'unscrubbed patient data', 1),
    ('PI cannot approve their own proposal', "does not hold 'review:approve' permission", 1),
    ('Dean R&D authorises publication', 'Dean R&D Authorizes Publication', 1),
    ('One-click rollback is demonstrated', 'One-Click Rollback Demonstration', 1),
    ('Finance clearance gate enforced above the threshold', 'Verification passed: Financial threshold control strictly enforced', 1),
    ('Periodic re-evaluation gate enforced', 'Verification passed: Periodic re-evaluation successfully enforced', 1)]

trace = console.getvalue()
universal = [
    ("Run produced console output", r"\S", 1),
    ("Graded Assignment 1 executed", r"\[Assignment 1\]", 1),
    ("Graded Assignment 2 executed", r"\[Assignment 2\]", 1),
]
results = []
for label, pattern, minimum in universal + CHECKS:
    found = len(re.findall(pattern, trace, flags=re.MULTILINE))
    results.append({"check": label, "pattern": pattern, "expected_min": minimum,
                    "found": found, "passed": found >= minimum})
no_errors = "Traceback (most recent call last)" not in trace
results.append({"check": "No unhandled exception in the run", "pattern": "Traceback",
                "expected_min": 0, "found": 0 if no_errors else 1, "passed": no_errors})

for i, r in enumerate(results, 1):
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {i:>2}. {r['check']}")
passed = sum(r["passed"] for r in results)
print(f"\nLab {LAB}: {passed}/{len(results)} acceptance checks passed")

outdir = Path(f"lab{LAB}_artifacts")
outdir.mkdir(exist_ok=True)
(outdir / f"lab{LAB}_console_trace.txt").write_text(trace, encoding="utf-8")
(outdir / f"lab{LAB}_acceptance_checks.json").write_text(
    json.dumps({"lab": LAB, "passed": passed, "total": len(results), "checks": results}, indent=2),
    encoding="utf-8")
print("Artifacts written to", outdir.as_posix())

assert passed == len(results), "acceptance checks failed"

[PASS]  1. Run produced console output
[PASS]  2. Graded Assignment 1 executed
[PASS]  3. Graded Assignment 2 executed
[PASS]  4. Proposal without a rollback plan is rejected
[PASS]  5. Unscrubbed patient data is rejected
[PASS]  6. PI cannot approve their own proposal
[PASS]  7. Dean R&D authorises publication
[PASS]  8. One-click rollback is demonstrated
[PASS]  9. Finance clearance gate enforced above the threshold
[PASS] 10. Periodic re-evaluation gate enforced
[PASS] 11. No unhandled exception in the run

Lab 12: 11/11 acceptance checks passed
Artifacts written to lab12_artifacts
